In [ ]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras.layers import Dropout, Dense, Flatten
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool, Healpy_Transformer

from mlpng import Core
from mlpng.utils import setup_logging, RMSELoss, rmse_metrics
from mlpng.utils.dataloaders import KappaDataset

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

In [ ]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {len(gpus)}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
core = Core(
    [
        "settings/n64.json",
        "--nsims",
        "10000",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)
shapes = core.shapes
phi_scale = 100

In [ ]:
batch_size = 32
max_epochs = 100
initial_LR = 5e-4

data_fraction = 0.1
split = np.array([0.8, 0.1, 0.1]) * data_fraction
duplicates = [25, 10, 5]

strategy = tf.distribute.MirroredStrategy()

In [ ]:
date_time = tf.timestamp().numpy().astype(int)
run_name = f"simple-{date_time}"
save_dir = f"{core.dirs['model']}/{core.name}"
keras_file = f"{save_dir}/encoder-fnl-{core.shapes_str}-{run_name}.keras"
cache_file = f"{core.name}/simple-{core.name}-f{data_fraction}"

os.makedirs(save_dir, exist_ok=True)
print(f"Run name: {run_name}")

print(f"Model file: {keras_file}")
print(f"Cache file: {cache_file}")

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
]

In [ ]:
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        K,
        pool_p,
        max_batch_size,
        dropout_rate=0.1,
        use_transformer=False,
        num_heads=8,
        transformer_layers=2,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.nside, self.npix, self.fin, self.fout = nside, npix, fin, fout
        self.K, self.pool_p, self.max_batch_size, self.dropout_rate = (
            K,
            pool_p,
            max_batch_size,
            dropout_rate,
        )

        layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation="gelu",
                use_bn=True,
                use_bias=True,
                initializer=tf.keras.initializers.HeNormal(),
            )
        ]
        if use_transformer:
            key_dim = min(1024, fout // num_heads)
            layers.append(
                Healpy_Transformer(
                    key_dim=key_dim,
                    num_heads=num_heads,
                    positional_encoding=True,
                    n_layers=transformer_layers,
                    activation="gelu",
                    layer_norm=True,
                )
            )
        layers.extend(
            [
                Dropout(dropout_rate),
                HealpyPool(pool_p, "AVG"),
            ]
        )

        self.body = HealpyGCNN(
            nside=nside,
            indices=np.arange(npix),
            layers=layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

    def call(self, x, training=False):
        return self.body(x, training=training)

In [ ]:
class EncoderFnlModel:
    def __init__(
        self,
        input_shape,
        n_outputs,
        max_batch_size=32,
        pool_p=2,
        transformer_levels="all",
        num_heads=8,
        transformer_layers=2,
        dropout_rate=0.1,
    ):
        self.input_shape = input_shape
        self.n_outputs = n_outputs
        self.max_batch_size = max_batch_size
        self.pool_p = pool_p
        self.transformer_levels = transformer_levels
        self.num_heads = num_heads
        self.transformer_layers = transformer_layers
        self.dropout_rate = dropout_rate
        self.npix = input_shape[1]
        self.nside = hp.npix2nside(self.npix)
        self.npol = input_shape[2]
        self.initializer = tf.keras.initializers.HeNormal()

    def get_model(self):
        nside_factor = 2**self.pool_p
        pixel_factor = 4**self.pool_p

        depth = int(math.log(self.nside, nside_factor))
        level_nsides = [self.nside // (nside_factor**i) for i in range(depth + 1)]
        level_npixels = [12 * ns**2 for ns in level_nsides]
        channels = [self.npol] + [2 ** (i + 5) for i in range(depth + 1)]
        Ks = [3] * (depth + 1)

        if self.transformer_levels == "all":
            use_transformer = [True] * depth
        else:
            use_transformer = [False] * depth
            for lvl in self.transformer_levels:
                idx = lvl if lvl >= 0 else depth + lvl
                if 0 <= idx < depth:
                    use_transformer[idx] = True

        print(
            f"Depth: {depth}, pool_p: {self.pool_p} ({pixel_factor}x pixel reduction)"
        )
        print(f"Level nsides: {level_nsides}")
        print(f"Level npixels: {level_npixels}")
        print(f"Channels: {channels}, Ks: {Ks}")
        print(
            f"Transformer at levels: {[i for i, t in enumerate(use_transformer) if t]}"
        )

        inputs = tf.keras.Input(shape=self.input_shape[1:], name="lensed")
        x = inputs

        for i in range(depth):
            x = EncoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=channels[i],
                fout=channels[i + 1],
                K=Ks[i],
                pool_p=self.pool_p,
                max_batch_size=self.max_batch_size,
                dropout_rate=self.dropout_rate,
                use_transformer=use_transformer[i],
                num_heads=self.num_heads,
                transformer_layers=self.transformer_layers,
            )(x)

        x = Flatten()(x)
        x = Dense(64, activation="sigmoid")(x)
        x = Dense(64, activation="sigmoid")(x)
        outputs = Dense(self.n_outputs)(x)

        return tf.keras.Model(inputs, outputs)

In [ ]:
logger.info("Creating dataset for fnl prediction")
ds = KappaDataset.fromCore(core, phi_scale=1000, x_output="lensed", y_output="fnl")

train, val, test = ds.split(
    train_size=split[0],
    val_size=split[1],
    test_size=split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=duplicates,
    shuffle=True,
    buffer_size=len(ds),
    rotate=[True, False, False],
    gaussian_mask_prob=[0.1, 0.0, 0.0],  # Only mask training data
    clear_cache=True,  # Remove existing cache files
    cache_file=cache_file,
    gen_batch_size=128,
)

# fnl_truth = np.concatenate([y for _, y in test])

In [ ]:
for _ in train:
    pass

In [ ]:
for _ in test:
    pass

In [ ]:
for _ in val:
    pass

In [ ]:
epoch_steps = math.ceil(core.total_sims * split[0] * duplicates[0] // batch_size)
decay_steps = epoch_steps  # * 2
use_cosine_decay = False  # Toggle: True = CosineDecayRestarts, False = ExponentialDecay

with strategy.scope():
    if use_cosine_decay:
        learning_rate = CosineDecayRestarts(
            initial_learning_rate=initial_LR,
            first_decay_steps=decay_steps,
            t_mul=2.0,
            m_mul=0.95,
            alpha=0.01,
        )
    else:
        learning_rate = ExponentialDecay(
            initial_learning_rate=initial_LR,
            decay_steps=decay_steps,
            decay_rate=0.96,
            staircase=True,
        )

    model = EncoderFnlModel(
        (None, core.npix, core.npols),
        n_outputs=len(shapes),
        max_batch_size=batch_size,
        pool_p=2,
        # transformer_levels=[-5, -4, -3, -2, -1],
        num_heads=8,
        transformer_layers=2,
        dropout_rate=0.1,
    ).get_model()

    model.compile(
        optimizer=AdamW(learning_rate, amsgrad=True, use_ema=True),
        loss="mse",  # RMSELoss(),
        metrics=rmse_metrics(shapes),
    )

model.summary()
print(f"\nModel parameters: {model.count_params():,}")

In [ ]:
logger.info("Training encoder fnl model")
history = model.fit(
    train, epochs=max_epochs, validation_data=val, callbacks=callbacks, verbose=1
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (RMSE)")
plt.title("Encoder fnl Model Training Loss")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## fnl Predictions on Test Set

In [ ]:
logger.info("Predicting fnl on test set")
fnl_preds = model.predict(test, verbose=0)

fnl_truth = np.concatenate([y for _, y in test])
fnl_truth_flat = fnl_truth.ravel()
fnl_preds_flat = fnl_preds.ravel()

fnl_error = fnl_preds_flat - fnl_truth_flat
print(f"True fnls shape: {fnl_truth_flat.shape}")
print(f"Predicted fnls shape: {fnl_preds_flat.shape}")
print(f"\nMetrics:")
print(f"Mean absolute error: {np.mean(np.abs(fnl_error)):.4f}")
print(f"RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

sigma = core.get_likelihoods(True)[0]
print(f"Sigma (likelihood bound): {sigma:.4f}")

In [ ]:
line = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=10)
axes[0].plot(line, line, "r--", label="Perfect prediction")
axes[0].plot(line, line + sigma, "g--", label=f"+σ={sigma:.1f}")
axes[0].plot(line, line - sigma, "g--", label=f"-σ={-sigma:.1f}")
axes[0].set_xlabel("True fnl")
axes[0].set_ylabel("Predicted fnl")
axes[0].set_title("fnl Prediction from Lensed Maps (Direct)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(fnl_error, bins=50, color="C0", alpha=0.8, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.1f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.1f}"
)
axes[1].set_xlabel("Prediction Error")
axes[1].set_ylabel("Count")
axes[1].set_title("fnl Error Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Layer Activation Visualization

In [ ]:
for x_sample, y_sample in test.take(1):
    test_input = x_sample[:1]
    test_target = y_sample[:1]
    break

print(f"Input shape: {test_input.shape}")
print(f"Target shape: {test_target.shape}")

layer_names = []
activations = []
x = test_input

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.InputLayer):
        layer_names.append("input")
        activations.append(x.numpy())
        continue

    try:
        x = layer(x, training=False)
        layer_names.append(layer.name)
        activations.append(x.numpy() if hasattr(x, "numpy") else np.array(x))
    except Exception as e:
        print(f"Skipping {layer.name}: {e}")

print(f"\nFound {len(activations)} layer activations:")
for name, act in zip(layer_names, activations):
    print(
        f"  {name}: shape={act.shape}, min={act.min():.4f}, max={act.max():.4f}, mean={act.mean():.4f}"
    )

In [ ]:
if len(activations) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    means = [act.mean() for act in activations]
    stds = [act.std() for act in activations]
    mins = [act.min() for act in activations]
    maxs = [act.max() for act in activations]
    x_pos = np.arange(len(layer_names))

    ax = axes[0, 0]
    ax.fill_between(x_pos, mins, maxs, alpha=0.3, label="Min-Max range")
    ax.plot(x_pos, means, "b-", linewidth=2, label="Mean")
    ax.fill_between(
        x_pos,
        np.array(means) - np.array(stds),
        np.array(means) + np.array(stds),
        alpha=0.3,
        color="blue",
        label="±1 Std",
    )
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Activation Value")
    ax.set_title("Activation Statistics Across Layers")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    variances = [act.var() for act in activations]
    ax.bar(x_pos, variances, alpha=0.7)
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Variance")
    ax.set_title("Activation Variance per Layer")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3, axis="y")

    ax = axes[1, 0]
    n_layers = len(activations)
    selected_indices = (
        [0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1]
        if n_layers > 5
        else list(range(n_layers))
    )
    colors = plt.cm.viridis(np.linspace(0, 1, len(selected_indices)))
    for idx, color in zip(selected_indices, colors):
        act = activations[idx].flatten()
        if len(act) > 10000:
            act = np.random.choice(act, 10000, replace=False)
        ax.hist(
            act,
            bins=50,
            alpha=0.5,
            color=color,
            label=f"{layer_names[idx]}",
            density=True,
        )
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Density")
    ax.set_title("Activation Distribution (Selected Layers)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    dead_threshold = 1e-6
    dead_percentages = [
        (np.abs(act) < dead_threshold).mean() * 100 for act in activations
    ]
    ax.bar(x_pos, dead_percentages, alpha=0.7)
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Percentage (%)")
    ax.set_title(f"Dead Neurons per Layer (<{dead_threshold})")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    print("\nLayer output ranges:")
    for i, (name, act) in enumerate(zip(layer_names, activations)):
        print(
            f"  {i:3d}. {name:30s}: [{act.min():+.4f}, {act.max():+.4f}], μ={act.mean():.4f}, σ={act.std():.4f}"
        )

In [ ]:
def plot_healpix_map(ax, hpx_map, title, cmap="viridis", nest=True, add_colorbar=True):
    if nest:
        hpx_map = hp.reorder(hpx_map, n2r=True)
    nside = hp.npix2nside(len(hpx_map))
    xsize, ysize = 800, 400
    theta = np.linspace(np.pi, 0, ysize)
    phi = np.linspace(-np.pi, np.pi, xsize)
    PHI, THETA = np.meshgrid(phi, theta)
    pix = hp.ang2pix(nside, THETA, PHI)
    grid_map = hpx_map[pix]
    im = ax.pcolormesh(phi, np.pi / 2 - theta, grid_map, cmap=cmap, shading="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)
    if add_colorbar:
        plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8)
    return im

In [ ]:
spatial_activations = [
    (name, act)
    for name, act in zip(layer_names, activations)
    if len(act.shape) == 3 and act.shape[1] >= 12
]

if len(spatial_activations) > 9:
    step = len(spatial_activations) // 8
    indices = (
        [0]
        + list(range(step, len(spatial_activations) - 1, step))
        + [len(spatial_activations) - 1]
    )
    indices = sorted(set(indices))[:9]
    spatial_activations = [spatial_activations[i] for i in indices]

n_plots = len(spatial_activations)
if n_plots > 0:
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        subplot_kw={"projection": "mollweide"},
    )
    axes = np.array(axes).flatten() if n_plots > 1 else [axes]

    for ax_idx, (name, act) in enumerate(spatial_activations):
        spatial_act = act[0, :, 0]
        npix = len(spatial_act)
        try:
            nside = hp.npix2nside(npix)
            plot_healpix_map(
                axes[ax_idx], spatial_act, f"{name}\n(nside={nside})", cmap="viridis"
            )
        except:
            axes[ax_idx].set_title(f"{name}\n(npix={npix})")

    for ax_idx in range(len(spatial_activations), len(axes)):
        axes[ax_idx].set_visible(False)

    plt.suptitle("Encoder Layer Activation Maps (Channel 0)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No spatial activations to visualize")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), subplot_kw={"projection": "mollweide"})

input_map = test_input[0, :, 0].numpy()
nside_in = hp.npix2nside(len(input_map))
plot_healpix_map(axes[0], input_map, f"Input (lensed)\nnside={nside_in}", cmap="RdBu_r")

if len(spatial_activations) > 0:
    last_name, last_act = spatial_activations[-1]
    last_map = last_act[0, :, 0]
    last_nside = hp.npix2nside(len(last_map))
    plot_healpix_map(
        axes[1],
        last_map,
        f"Final Spatial Layer\n({last_name}, nside={last_nside})",
        cmap="viridis",
    )
else:
    axes[1].set_title("No spatial activations")

plt.suptitle(
    f"Input → Bottleneck | Predicted fnl: {fnl_preds[0, 0]:.2f}, True: {fnl_truth[0, 0]:.2f}",
    fontsize=14,
)
plt.tight_layout()
plt.show()

## Gaussian Mask Verification Tests

In [ ]:
# Test 1: Verify fnl distribution - only training should have zeroed fnl values from gaussian_mask_prob
# Collect fnl values from each dataset split


def collect_fnl_values(dataset, name, max_batches=50):
    """Collect fnl values from dataset."""
    fnls = []
    for i, (_, y) in enumerate(dataset):
        if i >= max_batches:
            break
        fnls.append(y.numpy())
    return np.concatenate(fnls).ravel()


# Note: We need fresh datasets without cache to see the mask effect
# Using the existing train/val/test that were just created
train_fnls = collect_fnl_values(train, "train")
val_fnls = collect_fnl_values(val, "val")
test_fnls = collect_fnl_values(test, "test")

# Count zeros in each split
train_zeros = np.sum(train_fnls == 0)
val_zeros = np.sum(val_fnls == 0)
test_zeros = np.sum(test_fnls == 0)

print(
    f"Train: {len(train_fnls)} samples, {train_zeros} zeros ({100*train_zeros/len(train_fnls):.1f}%)"
)
print(
    f"Val:   {len(val_fnls)} samples, {val_zeros} zeros ({100*val_zeros/len(val_fnls):.1f}%)"
)
print(
    f"Test:  {len(test_fnls)} samples, {test_zeros} zeros ({100*test_zeros/len(test_fnls):.1f}%)"
)

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, fnls, name, expected_mask in zip(
    axes,
    [train_fnls, val_fnls, test_fnls],
    ["Train (mask_prob=0.1)", "Val (mask_prob=0.0)", "Test (mask_prob=0.0)"],
    [True, False, False],
):
    ax.hist(fnls, bins=50, alpha=0.7, edgecolor="black")
    zero_count = np.sum(fnls == 0)
    ax.axvline(0, color="r", linestyle="--", alpha=0.5)
    ax.set_xlabel("fnl value")
    ax.set_ylabel("Count")
    ax.set_title(f"{name}\nZeros: {zero_count} ({100*zero_count/len(fnls):.1f}%)")
    ax.grid(True, alpha=0.3)

plt.suptitle("fnl Distribution by Split - Gaussian Mask Verification", fontsize=14)
plt.tight_layout()
plt.show()

# Verification check
if train_zeros > 0 and val_zeros == 0 and test_zeros == 0:
    print("\n✅ PASS: Gaussian mask is correctly applied only to training data")
elif train_zeros == 0:
    print(
        "\n⚠️ WARNING: No zeros found in training - mask may not be working or cache was used"
    )
else:
    print(
        f"\n❌ FAIL: Found zeros in val ({val_zeros}) or test ({test_zeros}) - mask is leaking!"
    )

In [ ]:
# Test 2: Verify shuffle behavior - test dataset should be deterministic, train should shuffle


def get_first_n_samples(dataset, n=5):
    """Get first n samples from dataset as (x_hash, y) tuples for comparison."""
    samples = []
    for i, (x, y) in enumerate(dataset):
        if i >= n:
            break
        # Use hash of x data to identify samples (more robust than raw comparison)
        x_hash = hash(x.numpy().tobytes())
        samples.append((x_hash, y.numpy().copy()))
    return samples


# Test set should be deterministic (no shuffle, no reshuffle)
print("Testing test dataset determinism...")
test_epoch1 = get_first_n_samples(test, n=10)
test_epoch2 = get_first_n_samples(test, n=10)

test_same = all(
    h1 == h2 and np.allclose(y1, y2)
    for (h1, y1), (h2, y2) in zip(test_epoch1, test_epoch2)
)
print(f"Test set same order across iterations: {test_same}")

# Train set should shuffle between epochs (reshuffle=True by default)
print("\nTesting train dataset shuffling...")
train_epoch1 = get_first_n_samples(train, n=10)
train_epoch2 = get_first_n_samples(train, n=10)

train_different = not all(
    h1 == h2 for (h1, _), (h2, _) in zip(train_epoch1, train_epoch2)
)
print(f"Train set different order across iterations: {train_different}")

# Summary
print("\n" + "=" * 50)
if test_same and train_different:
    print("✅ PASS: Shuffle behavior is correct")
    print("   - Test set is deterministic (same order each epoch)")
    print("   - Train set is shuffled (different order each epoch)")
elif not test_same:
    print("❌ FAIL: Test set is NOT deterministic!")
elif not train_different:
    print(
        "⚠️ WARNING: Train set may not be shuffling (could be coincidence with small sample)"
    )